# Spam filter: train DistilBERT and measure every KPI

Runs the full training and evaluation for
[Email-Filtering-Using-DistilBERT](https://github.com/namita-tj/Email-Filtering-Using-DistilBERT) and gives you
one zip with all results to fill in the README.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.*

Then run the cells **one at a time, top to bottom** (Shift + Enter). Step 2 restarts the runtime once on purpose;
after that, carry on with step 3. Total time on a free T4: roughly 30–45 minutes, most of it in step 5.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf Email-Filtering-Using-DistilBERT
!git clone -q -b feature/hybrid-api https://github.com/namita-tj/Email-Filtering-Using-DistilBERT.git
%cd /content/Email-Filtering-Using-DistilBERT
!git log --oneline -1

## 2. Install compatible versions (then the runtime restarts)

Colab runs Python 3.13, which needs TensorFlow 2.20 or newer. This installs TensorFlow 2.20 with the matching
Keras 2 package (tf_keras) that Hugging Face's TensorFlow models need, plus the project's other dependencies.
The runtime then **restarts by itself** — you will see *"Your session crashed"* or *"Runtime restarted"*.
That is expected. Continue with step 3.

In [ ]:
%cd /content/Email-Filtering-Using-DistilBERT
!pip install -q "tensorflow==2.20.0" "tf_keras==2.20.1" "transformers==4.49.0" "datasets==3.3.2" "scikit-learn==1.6.1" "fastapi==0.115.6" "uvicorn==0.34.0" "pydantic==2.10.6" "openai==1.59.7" "boto3==1.37.9"
import os
os.kill(os.getpid(), 9)  # restart so the new versions are loaded

## 3. Check the setup

In [ ]:
%cd /content/Email-Filtering-Using-DistilBERT
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # Hugging Face TF models need Keras 2 (tf_keras), not Keras 3
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import sys, numpy, tensorflow as tf, transformers
print("Python      ", sys.version.split()[0])
print("TensorFlow  ", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "?")
print("Transformers", transformers.__version__, "| NumPy", numpy.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU         ", gpus or "NONE")
assert tf.__version__.startswith("2.20"), "Wrong TensorFlow version: run step 2 again."
assert gpus, ("No GPU visible. Runtime > Change runtime type > T4 GPU, then start again from step 1. "
              "If a GPU is selected and this still fails, run: !pip install -q 'tensorflow[and-cuda]==2.20.0' "
              "and restart the runtime.")

## 4. Optional: LLM fallback

The hybrid design sends uncertain emails to an LLM. To measure that part, add your OpenAI key in Colab:
click the **key icon** in the left sidebar → *Add new secret* → name `OPENAI_API_KEY` → paste the key →
switch on *Notebook access*. Cost for the full evaluation is a few cents.

Without a key, everything else is still measured; only the three LLM KPIs stay empty. Just run the cell.

In [ ]:
import os
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    key = None
if key:
    os.environ["OPENAI_API_KEY"] = key
    os.environ["LLM_PROVIDER"] = "openai"
    USE_LLM = True
    print("LLM fallback: ON (OpenAI)")
else:
    os.environ["LLM_PROVIDER"] = "none"
    USE_LLM = False
    print("LLM fallback: OFF (no OPENAI_API_KEY secret) - the LLM KPIs will be skipped")

## 5. Train (≈ 20–30 min)

Fine-tunes `distilbert-base-uncased` on all 28.5k Enron-Spam training emails, 2 epochs, 256 tokens per email.
Validation results and the LLM confidence band are printed at the end.

In [ ]:
import os
os.environ.update({"MAX_LENGTH": "256", "EPOCHS": "2", "TRAIN_LIMIT": "0", "TF_USE_LEGACY_KERAS": "1"})
!python -m training.train

## 6. Evaluate every KPI (≈ 10 min)

Test set (2,000 Enron emails), modern phishing emails as an out-of-distribution test, explainability
(Integrated Gradients, faithfulness, shortcut audit), latency and LLM cost.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
llm_flag = "" if USE_LLM else "--no-llm"
!python -m evaluation.run --phishing {llm_flag}

## 7. Results

In [ ]:
from IPython.display import Markdown, SVG, display
from pathlib import Path
display(Markdown(Path("reports/kpis.md").read_text()))
svg = Path("docs/explanation_example.svg")
if svg.exists():
    display(SVG(svg.read_text()))

## 8. Download the results

Downloads `spam_filter_results.zip` (reports, metrics, thresholds, the explanation image — no model weights).
**Send this zip to Claude** to fill in the README.

If the download does not start, open the folder icon on the left and right-click the zip → *Download*.

In [ ]:
!rm -f spam_filter_results.zip
!zip -q -r spam_filter_results.zip reports docs/explanation_example.svg spam_detection_model/metrics.json spam_detection_model/thresholds.json spam_detection_model/training_config.json
!ls -la spam_filter_results.zip
from google.colab import files
files.download("spam_filter_results.zip")

## 9. Optional: keep the trained model

The model (~270 MB) disappears when the Colab session ends. To keep it for the API or a demo, copy it to
Google Drive:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/spam_filter
!cp -r spam_detection_model /content/drive/MyDrive/spam_filter/
!ls -la /content/drive/MyDrive/spam_filter/spam_detection_model